# ParallelSearchRetriever

`ParallelSearchRetriever` is a LangChain [`BaseRetriever`](https://python.langchain.com/docs/concepts/retrievers/) backed by Parallel's [Search API](https://docs.parallel.ai/search-api/overview). It returns `list[Document]` with rich `metadata` (`url`, `title`, `publish_date`, `search_id`, `excerpts`, `query`) and slots into any RAG pipeline.

## Setup

In [ ]:
%pip install --quiet -U langchain-parallel

In [ ]:
import getpass
import os

if not os.environ.get("PARALLEL_API_KEY"):
    os.environ["PARALLEL_API_KEY"] = getpass.getpass("Parallel API key:\n")


## Basic retrieval

In [ ]:
from langchain_parallel import ParallelSearchRetriever

retriever = ParallelSearchRetriever(
    max_results=3,
    excerpts={"max_chars_per_result": 800},
)

docs = retriever.invoke("breakthroughs in fusion energy 2025")
for d in docs:
    print(d.metadata.get("title"), "-", d.metadata.get("url"))
    print(d.page_content[:200], "...\n")


Each `Document` has its excerpts joined into `page_content` and exposes the source URL, title, and publish date in `metadata` for downstream filters / citations.

## Configured search

Pass an `objective` for a richer retrieval objective alongside the keyword `search_queries`. The retriever forwards source/fetch policies to the underlying Search API.

In [ ]:
configured = ParallelSearchRetriever(
    max_results=5,
    excerpts={"max_chars_per_result": 1500},
    mode="basic",
    source_policy={
        "include_domains": ["nature.com", "science.org", "iter.org"],
    },
)

docs = configured.invoke(
    "What's the latest peer-reviewed result on net-energy-gain fusion?"
)
print(f"got {len(docs)} docs")
print("first url:", docs[0].metadata.get("url") if docs else "n/a")


## Async usage

In [ ]:
async def fetch():
    return await retriever.ainvoke("Latest GLP-1 trial results 2025")

docs = await fetch()
print(f"async got {len(docs)} docs")


## Drop into a RAG chain

The retriever plugs into any LangChain chain. Here's the simplest possible "retrieve → join → answer" wiring (replace `llm` with your tool-calling chat model — `ChatAnthropic`, `ChatOpenAI`, etc.). For a complete agent walkthrough, see [`docs/demo_agent.ipynb`](./demo_agent.ipynb).

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer using only the context below. Cite URLs."),
    ("human", "Context:\n{context}\n\nQuestion: {question}"),
])

def format_docs(docs):
    return "\n\n".join(f"[{d.metadata.get('url')}] {d.page_content[:500]}" for d in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
)

# Render the prompt for an example query (we don't bind an LLM here so the
# notebook stays self-contained for smoke tests).
filled = rag_chain.invoke("What was the most recent fusion energy breakthrough?")
print(filled.to_string()[:600], "...")


## API reference

- [Search API guides](https://docs.parallel.ai/search-api/overview)